# 0. les biblios

In [ ]:
import pandas as pd

import re

import matplotlib.pyplot as plt

import seaborn as sns

import numpy as np

# 1. Charger purchase_order_lines.csv

In [ ]:
df = pd.read_csv("purchase_order_lines.csv")

In [ ]:
print("Dimensions :", df.shape)

print("\nTypes de colonnes :")

df.info()

In [ ]:
print("\nPremières lignes :")

display(df.head())

- 651 lignes et 16 colonnes.

- Une ligne = une ligne de commande d'achat (`purchase_order_number`), passée par un site

(`ordering_facility_code`) auprès d'un fournisseur (`supplier_partner_code`) pour un article précis.

- C'est le fichier qui va relier `goods_receipts.csv` (ce qui a été reçu) à ce qui a été **commandé**.

# 2. Doublons

### a- Doublons de lignes

In [ ]:
print("Lignes strictement dupliquées :", df.duplicated().sum())

display(df[df.duplicated(keep=False)].sort_values('purchase_order_number'))

### b- Doublons sur l'identifiant métier

In [ ]:
print("purchase_order_number dupliqués :", df['purchase_order_number'].duplicated().sum())

- Solution

In [ ]:
df = df.drop_duplicates(keep='first')

df = df.drop_duplicates(subset='purchase_order_number', keep='first')



print("Nombre de lignes après suppression :", df.shape[0])

print("Doublons restants :", df['purchase_order_number'].duplicated().sum())

### c- Doublons de colonnes

In [ ]:
print("Noms de colonnes en double :", df.columns[df.columns.duplicated()].tolist())

print("Colonnes au contenu identique :", df.columns[df.T.duplicated()].tolist())

In [ ]:
print(df['purchase_order_line'].value_counts())

→ `purchase_order_line` ne contient qu'une seule valeur (`1`), comme dans `goods_receipts.csv` — même

remarque : pas de variance ici, mais utile comme clé de jointure future.

### d- Cohérence item_code ↔ item_description ↔ item_category

In [ ]:
print("Descriptions par code article :", df.groupby('item_code')['item_description'].nunique().value_counts())

print("Catégories par code article :", df.groupby('item_code')['item_category'].nunique().value_counts())

**Constat :** chaque `item_code` a toujours la même `item_description` (cohérent), mais **5 codes

articles ont 2 catégories différentes** — signe d'un problème de casse à corriger, pas une vraie

redondance (on vérifie ça à l'étape 3).

# 3. Formats incohérents

In [ ]:
print(df.dtypes)

### - Vérifier le format des identifiants <--->

In [ ]:
print("Format purchase_order_number :", df['purchase_order_number'].str.replace(r'\d', '9', regex=True).unique())

print("Format item_code :", df['item_code'].str.replace(r'[A-Z0-9]', 'X', regex=True).unique())

→ Formats homogènes.

### - Vérifier le format de order_date et requested_delivery_date (dates) <--->

In [ ]:
for col in ['order_date', 'requested_delivery_date']:

    dates_test = pd.to_datetime(df[col], errors='coerce')

    print(col, "-> non convertibles :", dates_test.isna().sum())

    display(df.loc[dates_test.isna(), col])

**Constat :** formats mixtes sur les deux colonnes de dates (`2022/10/01`, `10-Apr-2023`), comme sur

tous les fichiers du lot logistique — mais la majorité est au format ISO.



**Piège à éviter :** parser directement avec `format='mixed', dayfirst=True` sur toute la colonne casse

les dates ISO déjà propres (`2015-02-10` serait lu comme `2015-10-02`, inversant jour et mois). On parse

donc en deux temps : d'abord au format ISO strict, puis seulement les valeurs restées en échec sont

reparsées avec `dayfirst=True`.

- Solution

In [ ]:
def parse_dates_mixtes(colonne):

    dates = pd.to_datetime(colonne, format='ISO8601', errors='coerce')

    echec = dates.isna() & colonne.notna()

    dates[echec] = pd.to_datetime(colonne[echec], format='mixed', dayfirst=True, errors='coerce')

    return dates



df['order_date'] = parse_dates_mixtes(df['order_date'])

df['requested_delivery_date'] = parse_dates_mixtes(df['requested_delivery_date'])



print(df[['order_date', 'requested_delivery_date']].dtypes)

print("Non convertibles :", df[['order_date', 'requested_delivery_date']].isna().sum().sum())

print("Lignes où livraison demandée avant la commande :",

      (df['requested_delivery_date'] < df['order_date']).sum())

### - Vérifier le format de ordered_quantity (numérique) <--->

In [ ]:
s = df['ordered_quantity']

non_num = s[pd.to_numeric(s, errors='coerce').isna() & s.notna()]

print("Non convertibles :", len(non_num))

display(df.loc[non_num.index, ['ordered_quantity', 'ordered_unit']])

**Constat :** même schéma que `received_quantity` dans `goods_receipts.csv` — l'unité est parfois

recopiée dans la quantité (`"88 case"`), alors qu'elle existe déjà dans `ordered_unit`.

- Solution

In [ ]:
df['ordered_quantity'] = (df['ordered_quantity'].astype(str)

                           .str.replace(r'\s+[a-z_]+$', '', regex=True)

                           .str.strip())

df['ordered_quantity'] = pd.to_numeric(df['ordered_quantity'], errors='coerce')

print(df['ordered_quantity'].dtype)

print("Non convertibles restants :", df['ordered_quantity'].isna().sum())

### - Vérifier le format de unit_price (numérique) <--->

In [ ]:
s = df['unit_price']

non_num = s[pd.to_numeric(s, errors='coerce').isna() & s.notna()]

print("Non convertibles :", len(non_num), "| exemples :", non_num.unique())

**Constat :** virgule décimale, même correction que d'habitude.

In [ ]:
df['unit_price'] = pd.to_numeric(df['unit_price'].astype(str).str.replace(',', '.', regex=False), errors='coerce')

print(df['unit_price'].dtype)

print("Non convertibles restants :", df['unit_price'].isna().sum())

### - Vérifier le format de item_category (catégorielle) <--->

In [ ]:
print(sorted(df['item_category'].unique()))

**Constat :** 5 catégories écrites en MAJUSCULES (`MONITORING_EQUIPMENT`, `LABORATORY_MATERIAL`...) qui

correspondent en fait à des catégories déjà existantes en minuscules — c'est exactement la même casse

incohérente que sur les autres fichiers, ça confirme ce qu'on avait repéré à l'étape 2d.

- Solution

In [ ]:
df['item_category'] = df['item_category'].str.strip().str.lower()



print(sorted(df['item_category'].unique()))

print("Nombre de catégories après nettoyage :", df['item_category'].nunique())

print("Vérification : catégories par code article maintenant :",

      df.groupby('item_code')['item_category'].nunique().value_counts())

→ Confirmé : chaque code article n'a plus qu'une seule catégorie après nettoyage de la casse.

### - Vérifier les autres colonnes catégorielles

In [ ]:
for col in ['ordered_unit', 'currency', 'authorization_class', 'order_status']:

    print(col, "->", sorted(df[col].dropna().unique()))

    print()

→ Pas de souci de casse sur ces colonnes.

### - Cas particulier : currency varie parfois pour un même site

In [ ]:
print(df.groupby('ordering_facility_code')['currency'].nunique().value_counts())

display(df.groupby('ordering_facility_code')['currency'].unique().head(8))

**Constat :** plusieurs sites ont passé des commandes dans **plusieurs devises différentes**. Ce n'est

pas une incohérence de format à corriger — contrairement à `land_use_category` (un site ne change pas de

type de terrain), un site peut légitimement commander auprès de fournisseurs facturant dans des devises

différentes. **On ne touche pas à cette colonne**, c'est une vraie variation métier, pas une erreur.

# 4. Valeurs manquantes

In [ ]:
print("Valeurs manquantes par colonne :")

print(df.isna().sum())

Seule `declared_cost_center` a des manquants (3 lignes, 0.5%). On regarde s'il y a une logique

déductible, comme on l'a fait pour les fichiers précédents.

In [ ]:
display(df[df['declared_cost_center'].isna()][['ordering_facility_code', 'item_category', 'order_status']])

print()

print(df.groupby('item_category')['declared_cost_center'].apply(lambda s: s.dropna().nunique()))

**Constat :** un même `item_category` peut avoir plusieurs centres de coûts différents selon le site

ou le contexte de la commande — pas de règle fiable comme celle trouvée pour `fault_code` ou

`maintenance_provider`. Inventer un centre de coût serait un choix arbitraire.



**Décision** : on documente honnêtement l'absence, comme pour `inspection_status` dans

`goods_receipts.csv`.

In [ ]:
df['declared_cost_center'] = df['declared_cost_center'].fillna('unknown')



print(df['declared_cost_center'].value_counts())

print("Valeurs manquantes restantes :")

print(df.isna().sum())

# 5. Gestion des variables catégorielles

### a. Création de variables temporelles

In [ ]:
df['order_year'] = df['order_date'].dt.year

df['order_month'] = df['order_date'].dt.month

df['lead_time_days'] = (df['requested_delivery_date'] - df['order_date']).dt.days



display(df.head())

`lead_time_days` (délai entre la commande et la livraison demandée) est une nouvelle mesure utile,

calculée à partir des deux dates — elle pourra servir à comparer avec le délai réel une fois fusionné

avec `goods_receipts.csv` (date de commande vs. date de réception effective).

### b. Variables ordonnées

`authorization_class` (`ordinary` / `expedited` / `restricted` / `emergency`) a une vraie notion

d'urgence croissante, mais l'ordre exact n'est pas évident à trancher sans connaître le métier précisément

(`restricted` = accès limité, pas forcément "plus urgent" qu'`expedited` = accéléré). Par prudence, on la

garde nominale plutôt que d'imposer un ordre qu'on ne peut pas justifier avec certitude.

### c. Encodage des variables nominales : reporté après le merge

Comme pour les autres fichiers du lot logistique, on garde les catégories nettoyées en texte

(`item_category`, `ordered_unit`, `currency`, `authorization_class`, `declared_cost_center`,

`order_status`). L'encodage sera fait une seule fois après le merge avec `goods_receipts.csv`,

`erp_facilities.csv`, `erp_business_partners.csv`.

In [ ]:
df_stats = df.copy()

print(df_stats.dtypes)

display(df_stats.head())

# 6. Voir les outliers (colonnes numériques)

In [ ]:
cols_num = ['ordered_quantity', 'unit_price', 'lead_time_days']

display(df_stats[cols_num].describe().T)

In [ ]:
plt.figure(figsize=(14, 5))

for i, col in enumerate(cols_num, 1):

    plt.subplot(1, 3, i)

    sns.boxplot(y=df_stats[col], color="skyblue")

    plt.title(col)

plt.tight_layout()

plt.show()

In [ ]:
for col in cols_num:

    q1, q3 = df_stats[col].quantile([0.25, 0.75])

    iqr = q3 - q1

    n = ((df_stats[col] < q1 - 1.5*iqr) | (df_stats[col] > q3 + 1.5*iqr)).sum()

    print(f"{col:20s} min={df_stats[col].min():9.1f} max={df_stats[col].max():9.1f} -> {n} outliers")

**Constat, comme pour les autres fichiers logistiques :** `ordered_quantity` et `unit_price` mélangent

plusieurs unités très différentes (`service`, `pallet`, `grant`, `contract`...). Un `grant` ou un

`contract` a un prix unitaire très élevé (montant global) comparé à une `case` de consommables.

In [ ]:
display(df_stats.groupby('ordered_unit')[['ordered_quantity', 'unit_price']].mean().round(1))

→ Cohérent avec la nature de chaque unité. **On conserve toutes les valeurs.**

# 7. Histogrammes et distributions

In [ ]:
plt.figure(figsize=(14, 4))

for i, col in enumerate(cols_num, 1):

    plt.subplot(1, 3, i)

    sns.histplot(df_stats[col], kde=True, bins=25, color='skyblue')

    plt.title(f"Distribution : {col}")

plt.tight_layout()

plt.show()



print(df_stats[cols_num].skew().round(2))

→ `ordered_quantity` et `unit_price` sont très asymétriques (mélange d'unités, comme vu à l'étape 6).

`lead_time_days` est plus régulier, centré autour d'un délai standard.

In [ ]:
plt.figure(figsize=(14, 4))



plt.subplot(1, 3, 1)

sns.countplot(y='item_category', data=df_stats,

              order=df_stats['item_category'].value_counts().index, color='skyblue')

plt.title("Répartition par catégorie d'article")



plt.subplot(1, 3, 2)

sns.countplot(y='authorization_class', data=df_stats,

              order=df_stats['authorization_class'].value_counts().index, color='skyblue')

plt.title("Répartition des classes d'autorisation")



plt.subplot(1, 3, 3)

sns.countplot(y='order_status', data=df_stats,

              order=df_stats['order_status'].value_counts().index, color='skyblue')

plt.title("Répartition des statuts de commande")



plt.tight_layout()

plt.show()

# 8. Corrélations

In [ ]:
matrice_spearman = df_stats[cols_num].corr(method='spearman')

matrice_pearson = df_stats[cols_num].corr(method='pearson')

display(matrice_spearman.round(2))

In [ ]:
plt.figure(figsize=(6, 5))

sns.heatmap(matrice_spearman, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".2f", linewidths=0.5)

plt.title("Matrice de corrélation de Spearman")

plt.tight_layout()

plt.show()

In [ ]:
sns.pairplot(df_stats[cols_num], kind='scatter', plot_kws={'alpha': 0.3, 'color': 'teal', 's': 15})

plt.suptitle("Nuages de points croisés", y=1.02)

plt.show()

In [ ]:
print(df_stats.groupby('authorization_class')['lead_time_days'].mean().round(1))

### - Analyse des corrélations



- `ordered_quantity` et `unit_price` sont corrélées négativement (-0.62) : logique, les unités commandées

en grande quantité (`case`, `pack`) ont un prix unitaire plus faible que les unités "globales" achetées à

l'unité (`grant`, `contract`, `service`).

- `lead_time_days` ne corrèle avec rien de significatif, et sa moyenne est quasiment identique selon la

classe d'autorisation (23 à 25 jours dans tous les cas) — contrairement à l'intuition, `authorization_class`

ne semble pas piloter le délai de livraison demandé dans ce fichier. Ça confirme qu'on a bien fait de la

garder nominale plutôt que de lui imposer un ordre d'urgence supposé.

# 9. Export du jeu de données nettoyé

In [ ]:
df_stats.to_csv("purchase_order_lines_clean.csv", index=False)



print("Dimensions finales :", df_stats.shape)

print("Valeurs manquantes restantes :", df_stats.isna().sum().sum())

# 10. Synthèse du nettoyage



| Problème identifié | Colonnes concernées | Traitement appliqué |

|---|---|---|

| Doublon strict (1) et doublon d'identifiant | toutes | suppression, on garde la 1ère occurrence |

| Dates au format mixte (piège : `dayfirst` casse les dates ISO si mal utilisé) | `order_date`, `requested_delivery_date` | parsing en 2 temps : ISO strict, puis `mixed+dayfirst` uniquement pour les échecs |

| Unité recopiée dans la valeur | `ordered_quantity` | suppression de l'unité redondante |

| Virgule décimale | `unit_price` | `,` → `.` |

| Casse incohérente (5 catégories en double) | `item_category` | normalisation en minuscules, confirmé par la cohérence retrouvée avec `item_code` |

| 3 manquants sans logique déductible | `declared_cost_center` | remplacés par la catégorie explicite `unknown` |

| Variation légitime (pas une erreur) | `currency` | **conservée telle quelle** : un site peut commander dans plusieurs devises |

| Encodage des catégorielles | toutes | **reporté** : fait une seule fois après le merge des fichiers logistiques |



**Biais introduits et assumés :**

- `unknown` pour `declared_cost_center` évite d'inventer un centre de coût non déductible avec certitude ;

- `authorization_class` reste nominale malgré une hiérarchie d'urgence probable, faute de connaître avec

certitude l'ordre exact voulu par le métier ;

- comme pour les fichiers précédents, on garde la 1ère occurrence du doublon après vérification qu'il

était identique.